<a href="https://colab.research.google.com/github/sruthi-analyst/sruthi-codeboosters-2026/blob/main/Day8/Day_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#RAG: Retrieval Augmented Genertion.
Purpose: minimize the hallucination ,
##hallucination problem
   ->In the context of Retrieval Augmented Generation (RAG), hallucination refers to the generation of plausible-sounding but factually incorrect, unsupported, or contradictory information by the language model, even when provided with relevant external knowledge or documents. It's when the model 'makes things up' rather than strictly adhering to the provided facts.
(Plausible means seeming reasonable, probable, or likely to be true)
##Why this exists:
* Large Language Models(LLMs) have hard boundary: they know only what theu were trained on,
          they are prone to making things up (hallucinating) when they are pushed past that data.

* So, instead the data is stored(chromDB => converts into embedding) in rag system
* like an search engine is used to get the informations from the chromaDB so it gives only know values without hallucination.

##RAG Architecture: Pipeline:

###Phase A: Ingestion
1. Documents
2. Chunks
3. Embed
4. Store

###Phase B: Inference
1. User Qn
2. Embed Qn
3. Retrieve relevant chunks from vector store
4. Synthesize answer with LLM

Chunk - size 200 < chunk_size < 500 words

In [1]:
!pip install sentence-transformers chromadb groq pandas -q
#sentence-transformer: String to integer conversion
#chromadb:             Our vector db to store and search embeddings
#groq:                 Groq API client to call the LLM
#pandas                Loads and processes our CSV data
print("Installed library requirements")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

In [2]:
import pandas as pd #loading and managing college_notes.csv dataset
import chromadb # the vector db library to store document embeddings and perform similarity search
from sentence_transformers import SentenceTransformer #class that loads pre-trained LLM models
from groq import Groq #Groq API client for API calls
import os
from chromadb.utils import embedding_functions


In [3]:
import os;
from google.colab import userdata

try:
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("API Key successfully loaded from Colab Secrets!")
except userdata.SecretNotFoundError:
    print("Error: Please add 'GROQ_API_KEY' to your Colab Secrets sidebar.")

#os.environ stores the key as an environment variable
#This makes it accessible to the Groq library when it needs to authenticate.

#Initialise groq client
#Groq client is out connection to the Groq API services
groq_client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

print("Initialised GROQ API Key")
print("Note: If you see an authentication error later, double-check your API key.")

API Key successfully loaded from Colab Secrets!
Initialised GROQ API Key
Note: If you see an authentication error later, double-check your API key.


In [4]:
#this is our knowledge base - the documents our RAG system will retrieve from

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/datasets/college_notes.csv")
print("Shape of dataset: ", df.shape)
print("\nColumns: ", df.columns.tolist())
print("\nFirst 3 rows")
print(df.head(3))

Shape of dataset:  (15, 4)

Columns:  ['note_id', 'subject', 'topic', 'content']

First 3 rows
  note_id           subject          topic  \
0    N001  Data Engineering  ETL Pipelines   
1    N002  Data Engineering  SQL Databases   
2    N003  Data Engineering  Data Cleaning   

                                             content  
0  ETL stands for Extract Transform Load. It is t...  
1  A database is an organized collection of data ...  
2  Data cleaning involves fixing or removing inco...  


In [5]:
print("Subjects in the dataset:")
print(df['subject'].value_counts())

print("\nSample of topics:")
print(df[['note_id','subject', 'topic']].to_string(index=False))

print("\nLength od content (number of characters) for each note:")
df['content_length'] = df['content'].apply(len)
print(df[['topic', 'content_length']].to_string(index=False))

Subjects in the dataset:
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    2
Name: count, dtype: int64

Sample of topics:
note_id            subject                          topic
   N001   Data Engineering                  ETL Pipelines
   N002   Data Engineering                  SQL Databases
   N003   Data Engineering                  Data Cleaning
   N004   Data Engineering       APIs and Data Collection
   N005   Data Engineering           Big Data and PySpark
   N006   Machine Learning            Supervised Learning
   N007   Machine Learning               Model Evaluation
   N008   Machine Learning            Feature Engineering
   N009   Machine Learning                 Decision Trees
   N010   Machine Learning                  Random Forest
   N011      Generative AI          Large Language Models
   N012      Generative AI             Prompt Engineering
   N013      Generative AI Retrieval Augmented Generation
   N014 Python

In [6]:
documents = df['content'].to_list()
ids = [f"note_{row['note_id']}" for row in df.to_dict('records')]
metadata = [
    {'subject': row['subject'], 'topic': row['topic']}
    for row in df.to_dict('records')
]

print(f"Total chunks prepared: {len(documents)}")
print(f"First document ID: {ids[0]}")
print(f"First metadata: {metadata[0]}")
print(f"First document: {documents[0][:100]}...")

Total chunks prepared: 15
First document ID: note_N001
First metadata: {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
First document: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc...


In [7]:
#Loading embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
test_embedding = embedding_model.encode("This is a test sentence.")
print(f"Test embedding shape: {test_embedding.shape}")
print("First 5 values of test embedding:")
print(test_embedding[:5])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Test embedding shape: (384,)
First 5 values of test embedding:
[0.08429647 0.05795366 0.00449333 0.1058211  0.00708344]


In [8]:
#chromadb.client: Creates a Croma DB clientb that stores data in memory
#In memory means data exist only while the notebook is running.
#For permanent storage you would use chromadb.persistentClient(path='/.croma_db')
chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(name='college_notes_rag')
# collection in a Croma DB is like a table in the regular database.
#It groups related to documents together
#get_or_create_collections(): creation new collection if it is not existing one if it.

print("chromaDB client created")
print("Collection name: college_notes_rag")
print(f"Documents in collection so far: {collection.count()}")

chromaDB client created
Collection name: college_notes_rag
Documents in collection so far: 0


In [9]:
print("Generating embeddings for all 15 notes...")

embeddings = embedding_model.encode(documents, show_progress_bar=True)
print(f"\nEmbedding matrix shape: {embeddings.shape}")
embeddings_list = embeddings.tolist()

collection.add(
    documents=documents,
    metadatas=metadata,
    ids=ids,
    embeddings=embeddings_list
)
print("\nDOcuments successfully added to chromaDB")
print(f"\nTotal documents in collection: {collection.count()}")


Generating embeddings for all 15 notes...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding matrix shape: (15, 384)

DOcuments successfully added to chromaDB

Total documents in collection: 15


In [10]:
"""
  Given a user question, retrieve the most relevant document chunks form ChromaDB.

  Parameters:
      question (str) : The user's questions as a text string
      top_k     (int) : How many top results to return (default: 3)

  Returns:
      A dictionary containing the retrived documents, distances and their metadata
"""
def retrieve_relevant_chunks(question, top_k=3):
  question_embedding = embedding_model.encode(question).tolist()
  results = collection.query(
      query_embeddings=[question_embedding],
      n_results=top_k
  )

  return results

print("Retrival function defined successfully!")
print("Function: retrieve_relevant_chunks(question, top_k=3)")

Retrival function defined successfully!
Function: retrieve_relevant_chunks(question, top_k=3)


In [11]:
test_question="What is ETL"
results = retrieve_relevant_chunks(test_question, top_k=3)
print("\nTop 3 Retrieved Chunks:")
print('-'*30)
print(results['documents'][0][0])
print('-'*30)
print(results['documents'][0][1])
print('-'*30)
print(results['documents'][0][2])
print('-'*30)



Top 3 Retrieved Chunks:
------------------------------
ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.
------------------------------
An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services like weather data stock prices or social media feeds.
------------------------------
RAG or Retrieval Augmented Generation is a technique where an AI model first retrieves relevant documents from a knowledge base and then generates an answer based on those retrieved documents. This reduces hallucination and allows AI to answer questions about specific data.
------------------------------


In [12]:
for i, (doc, dist, meta) in enumerate(zip(results['documents'][0], results['distances'][0], results['metadatas'][0])):
  print(f"Document {i+1}")
  print(f"\nResult {i+1}:")
  print(f"  Subject   : {meta['subject']}")
  print(f"  Topic    : {meta['topic']}")
  print(f"  Distance : {dist}")
  print(f"  Content  : {doc[:100]}...")

Document 1

Result 1:
  Subject   : Data Engineering
  Topic    : ETL Pipelines
  Distance : 0.3394540250301361
  Content  : ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc...
Document 2

Result 2:
  Subject   : Data Engineering
  Topic    : APIs and Data Collection
  Distance : 1.5857576131820679
  Content  : An API or Application Programming Interface allows two software applications to talk to each other. ...
Document 3

Result 3:
  Subject   : Generative AI
  Topic    : Retrieval Augmented Generation
  Distance : 1.6214964389801025
  Content  : RAG or Retrieval Augmented Generation is a technique where an AI model first retrieves relevant docu...


In [13]:
def build_context_from_results(results):
  context_parts = []

  for i, (doc, meta) in enumerate(zip(
      results['documents'][0],
      results['metadatas'][0]
  )):
    context_text = f"[Source {i+1}: {meta['subject']} - {meta['topic']}\n{doc}]"
    context_parts.append(context_text)

    return context_parts

In [14]:
context_built = build_context_from_results(results=results)
for i, context in enumerate(context_built):
  print(f"Context {i+1}:\n{context}\n")

Context 1:
[Source 1: Data Engineering - ETL Pipelines
ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.]



In [15]:
def generate_rag_answer(question, context):
  """
  Send the retrieved context and question to the Groq LLM for answe generation.

  Parameters:
    question (str): The user's question
    context (str) : The retrieved context chunks (formatted string)

  Returnd:
    answer (str) : The LLM's generated answer
  """
  #SYSTEM PROMPT: Instructions to the LLM about its role and behavior
  #This is the key to RAG - we tell the LLM to ONLY use the provided context
  system_prompt = """Your are a helpful academic assistant for engineering students.

  You will be given context retrieved from a college knowledge base, and a student's question

  RULES:
  1. Answer ONLY using the instruction provided in the context below.
  2. If the answer is not found in the context, say exactly:
      "I don't have enough information in my knowledge base to answer this question."
  3. Do not use your general training knowledge.
  4. Keep answer clear, accurate, and beginner-friendly.
  5. Mention which source the information came from when possible."""

  #USER PROMPT: The context + question formatted
  user_prompt = f"""Context:
  {context}

  Question:
  {question}
  Please answer the question based only on the context provided above."""

  response = groq_client.chat.completions.create(
      model="llama-3.1-8b-instant", #to use same model from training
      messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_prompt}
      ],
      temperature = 0.1,
      #temperature - 0.1 - Very low randomness - we want factual, consistent answers, for RAG, low temp is preffered so the LLM sticks to the context
      max_tokens = 500 #max len of gen responce
  )

  #Extract the text answer from the API resonse object
  answer = response.choices[0].message.content
  #response.choices : A list of responses
  return answer
print("Defined RAG generation function")

Defined RAG generation function


In [16]:
def ask_college_assistant(question, top_k=3, verbose=True):
    """
    Parameters:
      question (str) : The user's question
      top_k    (int): Returns the top 3 chunks from the collection
      verbose (bool): Whether to print intermediate steps (True)

    Returns:
      answer (str): The final generated answer
    """
    if verbose:
      print(f"Question: {question}")
      print("="*60)
      print("Step 1: Retrieving Relevant Chunks...")
      print("="*60)

    results = retrieve_relevent_chunks(question, top_k=top_k)

    if verbose:
      print("Step 2: Building Context...")
      print("="*60)
    context = build_context_from_results(results)

    if verbose:
      print("Step 3: Generating Answer...")
      print("="*60)
    answer = generate_rag_answer(question, context)

    return answer

### **Putting it all together: The RAG Pipeline**

1. Load college_notes.csv
2. Embed all notes and index them in ChromaDB
3. Accept any student question as input
4. Retrieve the top 3 relevant notes using vector similarity
5. Inject retrieved chunks as context into Groq LLM prompt
6. Generate a clear grounded response that cites the content

In [17]:
HISTORY = []

In [23]:
student_question = input("Enter your question: ")

# 1. Retrieve relevant chunks
retrieved_results = retrieve_relevant_chunks(student_question, top_k=3)

# 2. Build context from retrieved chunks
context_for_llm = build_context_from_results(results=retrieved_results)

# Join the context parts into a single string for the LLM
formatted_context = "\n\n".join(context_for_llm)

# 3. Generate the answer using the RAG function
final_answer = generate_rag_answer(student_question, formatted_context)

HISTORY.append([student_question, final_answer])

print(f"Question: {student_question}")
print("\n--- Generated Answer ---")
print(final_answer)

Enter your question: Is Machine Learning and GenAI the same?
Question: Is Machine Learning and GenAI the same?

--- Generated Answer ---
Based on the provided context, it appears that Generative AI (GenAI) and Large Language Models (LLM) are related concepts, but not exactly the same as Machine Learning.

The context mentions that a Large Language Model (LLM) is a type of AI model, but it does not explicitly state that Machine Learning and GenAI are the same. However, it does mention that Machine Learning is not explicitly mentioned in the context.

Therefore, I would say that Machine Learning and GenAI are not the same based on the provided context.


In [24]:
HISTORY

[['What is ETL?',
  'ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources, transforming it into a clean and structured format, and loading it into a database or data warehouse for analysis. \n\n[Source: Data Engineering - ETL Pipelines]'],
 ['current gold price?',
  "I don't have enough information in my knowledge base to answer this question."],
 ['Explain API',
  'An API, or Application Programming Interface, is a way for two software applications to talk to each other. In data engineering, APIs are used to fetch data from external services, such as weather data, stock prices, or social media feeds. \n\n[Source: Data Engineering - APIs and Data Collection]'],
 ['Is Machine Learning and GenAI the same?',
  'Based on the provided context, it appears that Generative AI (GenAI) and Large Language Models (LLM) are related concepts, but not exactly the same as Machine Learning.\n\nThe context mentions that a Large Language Model (LLM) is a 